# Wire FAISS Retrieval → Mistral 7B Chatbot (GPU)

This notebook connects your **FAISS vector store** (from Step 2) to the **Mistral 7B Instruct** generator (from Step 3).
You’ll get end‑to‑end RAG: query → retrieve top‑k chunks → token‑safe prompt → answer + clean sources.

**Requirements**
- Files from Step 2: `outputs/faiss_index.bin`, `outputs/metadata.json`
- GPU environment with PyTorch + `transformers`

> If you see OOM errors, reduce `k` or switch to a smaller model (e.g., `mistralai/Mistral-7B-Instruct-v0.1` or use 4-bit).


In [ ]:
# If needed, install dependencies
# !pip install sentence-transformers transformers faiss-cpu accelerate bitsandbytes


In [1]:
import os, json, re
from pathlib import Path
from typing import List, Tuple, Dict, Any

import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline


In [2]:
# Paths to artifacts from 2
base_path = Path(r"C:\Users\ADMIN\Documents\rag_chatbot")
index_path = base_path / "faiss_index.bin"
metadata_path= base_path / "metadata.json"

assert index_path.exists(), f"Missing {index_path}. Run Step 2 first."
assert metadata_path.exists(),  f"Missing {metadata_path}. Run Step 2 first."

# Load FAISS + metadata
index = faiss.read_index(str(index_path))
with open(metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print(f"Loaded FAISS with {index.ntotal} vectors; metadata records: {len(metadata)}")


Loaded FAISS with 607 vectors; metadata records: 607


In [3]:
# Embeddings (use the same model as Step 2 to avoid mismatch)
EMBED_MODEL = os.getenv("HF_EMBED_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
embedder = SentenceTransformer(EMBED_MODEL)

def embed_query(q: str) -> np.ndarray:
    return embedder.encode([q], convert_to_numpy=True).astype("float32")


In [4]:
# Stronger chat model on GPU 
GEN_MODEL = os.getenv("HF_GEN_MODEL", "mistralai/Mistral-7B-Instruct-v0.2")

tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL)
# Tip: to save VRAM, you can load 4-bit quantized via bitsandbytes; uncomment below:
# from transformers import BitsAndBytesConfig
# quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype="float16")
# model = AutoModelForCausalLM.from_pretrained(GEN_MODEL, device_map="auto", quantization_config=quant_config)
# Otherwise, default full precision/auto:
model = AutoModelForCausalLM.from_pretrained(GEN_MODEL, device_map="auto", torch_dtype="auto")

generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

print("Loaded generator:", GEN_MODEL)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Device set to use cpu


Loaded generator: mistralai/Mistral-7B-Instruct-v0.2


In [5]:
SYSTEM_PROMPT = (
    "You are a helpful assistant. Answer ONLY using the provided context. "
    "If the answer is not in the context, say you don't know. "
    "Write clearly. After the answer, list unique sources as: Sources: (file1.pdf), (file2.pdf)."
)

def _norm_source_path(p: str | None) -> str | None:
    if not p:
        return None
    p2 = p.replace("\\", "/")
    return p2.split("/")[-1]

def token_len(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=False))

def build_token_safe_context(question: str, idxs: List[int], reserve_tokens: int = 128) -> tuple[str, List[Dict[str, Any]]]:
    max_inp = tokenizer.model_max_length
    sys_toks = token_len(SYSTEM_PROMPT)
    q_toks   = token_len(question)
    budget   = max_inp - sys_toks - q_toks - reserve_tokens
    if budget <= 0:
        raise ValueError(f"Token budget too small. system={sys_toks}, question={q_toks}, max={max_inp}")
    
    parts: List[str] = []
    chosen: List[Dict[str, Any]] = []
    remaining = budget

    for rank, idx in enumerate(idxs, start=1):
        md = metadata[str(idx)] if isinstance(metadata, dict) and str(idx) in metadata else metadata[idx]
        txt = md.get("text") if isinstance(md, dict) else md["text"]
        src = None
        if isinstance(md, dict):
            mmeta = md.get("metadata", {}) or {}
            src = mmeta.get("source") or mmeta.get("source_path")
        src = _norm_source_path(src)

        header = f"[{rank}] (source: {src})\n"
        piece  = header + txt + "\n\n"
        ptoks  = token_len(piece)

        if ptoks <= remaining:
            parts.append(piece)
            chosen.append({"rank": rank, "index": idx, "source": src, "truncated": False})
            remaining -= ptoks
        else:
            if remaining > 10:
                ids = tokenizer.encode(piece, truncation=True, max_length=remaining, add_special_tokens=False)
                trunc_text = tokenizer.decode(ids, skip_special_tokens=True)
                if header.strip() not in trunc_text:
                    trunc_text = header + trunc_text
                parts.append(trunc_text)
                chosen.append({"rank": rank, "index": idx, "source": src, "truncated": True})
            break

    return "".join(parts), chosen

def retrieve(q: str, k: int = 6) -> Tuple[List[int], List[float]]:
    vec = embed_query(q)
    D, I = index.search(vec, min(k, index.ntotal))
    return I[0].tolist(), D[0].tolist()

def make_prompt(context: str, question: str) -> str:
    return f"""{SYSTEM_PROMPT}

Context:
{context}

Question: {question}

Answer (then list Sources with filenames only):"""


In [6]:
def answer(question: str, k: int = 6, max_new_tokens: int = 384, temperature: float = 0.2) -> Dict[str, Any]:
    idxs, dists = retrieve(question, k=k)
    context, chosen = build_token_safe_context(question, idxs, reserve_tokens=128)
    prompt = make_prompt(context, question)

    out = generator(prompt, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature, top_p=0.9)[0]["generated_text"].strip()

    # Ensure we append clean Sources even if model forgets
    seen = []
    for c in chosen:
        src = c.get("source")
        if src and src not in seen:
            seen.append(src)
    if "Sources:" not in out:
        out += "\n\nSources: " + (", ".join(f"({s})" for s in seen) if seen else "(N/A)")
    return {"answer": out, "sources": chosen, "indices": idxs}


### Quick test

Ask a question about your PDFs (requires Step 2 artifacts present).

In [7]:
resp = answer("Give me a brief summary of the document(s).", k=6)
print(resp["answer"])
print("\nDebug sources:")
for s in resp["sources"]:
    print(f"[{s['rank']}] idx={s['index']} src={s['source']} truncated={s['truncated']}")


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


You are a helpful assistant. Answer ONLY using the provided context. If the answer is not in the context, say you don't know. Write clearly. After the answer, list unique sources as: Sources: (file1.pdf), (file2.pdf).

Context:
[1] (source: admm_distr_stats.pdf)
ature; here, we limit ourselves to a basic but still very general result
that applies to all of the examples we will consider. We will make one

[2] (source: admm_distr_stats.pdf)
believe it should be.
The main contributions of this review can be summarized as follows:
(1) We provide a simple, cohesive discussion of the extensive
literature in a way that emphasizes and uniﬁes the aspects
of primary importance in applications.

[3] (source: admm_distr_stats.pdf)
able means that
(xi)j = zG(i,j),i =1 ,...,N, j =1 ,...,n i.
If G(i,j )= j for all i, then each local variable is just a copy of
the global variable, and consensus reduces to global variable consen-
sus, x
i = z. General consensus is of interest in cases where ni ≪ n,
so 